In [1]:

# ============================================================
# WORKLOADWISE AI
# Employee Productivity vs. Workload Intelligence Agent
# Fictional Uber Operations | Synthetic Data Only
# Google Colab - Run this entire cell
# ============================================================

%pip -q install gradio pandas plotly numpy

import pandas as pd
import numpy as np
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go

# ------------------------------------------------------------
# 1. CREATE SYNTHETIC EMPLOYEE DATA
# ------------------------------------------------------------

np.random.seed(42)

teams = {
    "Driver Support": 25,
    "City Operations": 25,
    "Customer Experience": 25,
    "Safety Operations": 20,
    "Marketplace Operations": 25
}

rows = []
employee_number = 1

for team, count in teams.items():
    for _ in range(count):
        workload = np.random.randint(25, 61)
        overtime = max(0, round(np.random.normal(7, 5), 1))
        absenteeism = max(0, round(np.random.exponential(1.8), 1))
        engagement = int(np.clip(np.random.normal(7, 1.7), 1, 10))

        # Higher overtime/workload can be associated with
        # lower completion and performance in this synthetic data.
        overload = max(0, workload - 40) * 0.65
        completion = np.clip(
            96 - overload - overtime * 0.75
            - absenteeism * 1.8
            + np.random.normal(0, 7),
            35, 100
        )

        performance = np.clip(
            45 + completion * 0.38
            + engagement * 1.2
            - overtime * 0.45
            + np.random.normal(0, 5),
            35, 100
        )

        rows.append({
            "Employee ID": f"UB{employee_number:03d}",
            "Team": team,
            "Weekly Work Hours": workload,
            "Overtime Hours": overtime,
            "Absence Days": absenteeism,
            "Task Completion (%)": round(completion, 1),
            "Engagement Score": engagement,
            "Performance Score": round(performance, 1)
        })
        employee_number += 1

employee_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# 2. BUILD THE WORKLOAD INTELLIGENCE AGENT
# ------------------------------------------------------------

def analyze_workforce(team_filter="All Teams"):
    data = employee_df.copy()

    if team_filter != "All Teams":
        data = data[data["Team"] == team_filter].copy()

    if data.empty:
        return "No employee records found.", data, None, data, "No recommendations available."

    # Transparent risk indicators; these are operational flags,
    # not diagnoses or definitive judgments about employees.
    data["High Workload"] = data["Weekly Work Hours"] > 45
    data["High Overtime"] = data["Overtime Hours"] > 10
    data["Absence Flag"] = data["Absence Days"] >= 3
    data["Low Completion"] = data["Task Completion (%)"] < 75
    data["Low Engagement"] = data["Engagement Score"] <= 5
    data["Low Performance"] = data["Performance Score"] < 70

    data["Risk Score"] = (
        data["High Workload"].astype(int) * 2
        + data["High Overtime"].astype(int) * 2
        + data["Absence Flag"].astype(int)
        + data["Low Completion"].astype(int) * 2
        + data["Low Engagement"].astype(int)
        + data["Low Performance"].astype(int)
    )

    def risk_label(score):
        if score >= 6:
            return "High"
        elif score >= 3:
            return "Moderate"
        return "Low"

    data["Operational Risk"] = data["Risk Score"].apply(risk_label)

    # Employee-level intervention recommendations
    def recommend(row):
        actions = []

        if row["Weekly Work Hours"] > 45:
            actions.append("Review workload and redistribute tasks")
        if row["Overtime Hours"] > 10:
            actions.append("Reduce recurring overtime; review staffing")
        if row["Task Completion (%)"] < 75:
            actions.append("Check blockers, task complexity and deadlines")
        if row["Absence Days"] >= 3:
            actions.append("Review absence-related coverage needs")
        if row["Engagement Score"] <= 5:
            actions.append("Arrange a supportive manager check-in")
        if row["Performance Score"] < 70:
            actions.append("Review goals, resources and coaching needs")

        if not actions:
            actions.append("Maintain workload balance and monitor trends")

        return "; ".join(actions)

    data["Recommended Intervention"] = data.apply(recommend, axis=1)

    # KPI calculations
    avg_hours = data["Weekly Work Hours"].mean()
    avg_overtime = data["Overtime Hours"].mean()
    avg_completion = data["Task Completion (%)"].mean()
    avg_engagement = data["Engagement Score"].mean()
    high_risk_count = (data["Operational Risk"] == "High").sum()
    overload_count = data["High Workload"].sum()
    overtime_count = data["High Overtime"].sum()
    low_completion_count = data["Low Completion"].sum()

    # Operational summary
    summary = f"""
# WorkloadWise AI — Workforce Report

### Scope
- **Employees analyzed:** {len(data)}
- **Team selected:** {team_filter}

### Key performance indicators
| Metric | Result |
|---|---:|
| Average weekly work hours | {avg_hours:.1f} hours |
| Average overtime | {avg_overtime:.1f} hours |
| Average task completion | {avg_completion:.1f}% |
| Average engagement | {avg_engagement:.1f}/10 |
| Employees flagged high risk | {high_risk_count} |
| Employees with high workload | {overload_count} |
| Employees with high overtime | {overtime_count} |
| Employees with low task completion | {low_completion_count} |

### Agent interpretation
"""

    if avg_hours > 45 and avg_completion < 80:
        summary += (
            "\n**Potential workload bottleneck:** Working hours are elevated "
            "while task completion is comparatively low. Review workload "
            "distribution, task complexity and staffing capacity."
        )
    elif avg_overtime > 10 and avg_completion < 80:
        summary += (
            "\n**Potential overtime bottleneck:** High overtime alongside "
            "low completion may indicate capacity constraints or process "
            "inefficiencies."
        )
    elif avg_engagement < 6:
        summary += (
            "\n**Potential engagement concern:** Review workload fairness, "
            "manager support and operational friction through supportive "
            "employee feedback."
        )
    else:
        summary += (
            "\n**Overall view:** Review team-level differences and the "
            "employee flags below to identify where targeted intervention "
            "may be useful."
        )

    summary += (
        "\n\n*These are synthetic records and screening indicators. "
        "Flags identify patterns for human review; they do not establish "
        "the cause of performance changes or evaluate employee wellbeing.*"
    )

    # Team comparison chart
    team_summary = (
        data.groupby("Team", as_index=False)
        .agg({
            "Weekly Work Hours": "mean",
            "Overtime Hours": "mean",
            "Task Completion (%)": "mean",
            "Engagement Score": "mean",
            "Performance Score": "mean"
        })
    )

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=team_summary["Team"],
        y=team_summary["Weekly Work Hours"],
        name="Weekly Work Hours",
        yaxis="y"
    ))
    fig.add_trace(go.Scatter(
        x=team_summary["Team"],
        y=team_summary["Task Completion (%)"],
        name="Task Completion (%)",
        mode="lines+markers",
        yaxis="y2"
    ))
    fig.update_layout(
        title="Team Workload vs. Task Completion",
        xaxis_title="Operations Team",
        yaxis=dict(title="Average weekly hours"),
        yaxis2=dict(
            title="Task completion (%)",
            overlaying="y",
            side="right",
            range=[0, 105]
        ),
        legend=dict(orientation="h", y=-0.35),
        height=430,
        margin=dict(b=120)
    )

    # High-priority employee records for human review
    priority = data[data["Operational Risk"] == "High"].copy()
    priority = priority.sort_values(
        ["Risk Score", "Overtime Hours"], ascending=False
    )

    display_columns = [
        "Employee ID", "Team", "Weekly Work Hours", "Overtime Hours",
        "Absence Days", "Task Completion (%)", "Engagement Score",
        "Performance Score", "Operational Risk", "Risk Score",
        "Recommended Intervention"
    ]

    data = data.sort_values("Risk Score", ascending=False)

    if priority.empty:
        priority_message = (
            "No employees met the high-risk screening threshold in this "
            "selection. Continue monitoring workload and team-level trends."
        )
    else:
        priority_message = (
            f"## Priority review list\n\n"
            f"{len(priority)} employee record(s) meet the high-risk "
            "screening threshold. Review the operational context with "
            "appropriate managers before making changes.\n\n"
            "Recommended first steps:\n"
            "- Check whether workload is unevenly distributed.\n"
            "- Review recurring overtime and staffing coverage.\n"
            "- Identify task blockers, dependencies and unrealistic deadlines.\n"
            "- Ask employees what support or changes would help.\n"
            "- Reassess indicators after interventions."
        )

    return (
        summary,
        data[display_columns],
        fig,
        priority[display_columns],
        priority_message
    )


# ------------------------------------------------------------
# 3. CREATE A USER-FRIENDLY GRADIO INTERFACE
# ------------------------------------------------------------

custom_css = """
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css,
    title="WorkloadWise AI"
) as demo:

    gr.Markdown("""
    # WorkloadWise AI
    ### Employee Productivity & Workload Intelligence

    **Fictional Uber Operations | Synthetic Data Demonstration**

    Understand workload patterns, discover possible operational
    bottlenecks, and generate recommendations for human review.
    """)

    with gr.Row():
        team_dropdown = gr.Dropdown(
            choices=["All Teams"] + list(teams.keys()),
            value="All Teams",
            label="Choose an operations team",
            interactive=True
        )
        analyze_button = gr.Button(
            "Analyze Workforce",
            variant="primary"
        )

    with gr.Tabs():
        with gr.Tab("Executive Overview"):
            summary_output = gr.Markdown()
            chart_output = gr.Plot(label="Workload Analytics")

        with gr.Tab("Employee Insights"):
            gr.Markdown(
                "Employee-level synthetic records and suggested "
                "operational interventions. Use these flags to guide "
                "human review, not as automatic employment decisions."
            )
            employee_table = gr.Dataframe(
                label="Workforce Analysis",
                interactive=False,
                wrap=True
            )

        with gr.Tab("Priority Reviews"):
            priority_message_output = gr.Markdown()
            priority_table = gr.Dataframe(
                label="Records Requiring Review",
                interactive=False,
                wrap=True
            )

        with gr.Tab("How It Works"):
            gr.Markdown("""
            ### Agent workflow

            1. **Collect data:** Load synthetic employee and operations metrics.
            2. **Measure workload:** Review weekly hours and overtime.
            3. **Measure delivery:** Review task completion and performance.
            4. **Review context:** Consider absenteeism and engagement.
            5. **Flag patterns:** Apply transparent screening thresholds.
            6. **Identify bottlenecks:** Compare workload with task completion.
            7. **Generate interventions:** Recommend workload redistribution,
               staffing review, blocker resolution and manager check-ins.
            8. **Present results:** Show KPIs, charts and employee-level findings.
            9. **Support managers:** Help prioritize questions for further review.
            10. **Monitor outcomes:** Reassess metrics after changes.

            ### Important limitations
            - All records are artificially generated for demonstration.
            - This is a rule-based prototype, not a trained predictive AI model.
            - Correlations do not prove that overtime or workload caused lower productivity.
            - Engagement and absence indicators should be handled respectfully.
            - Do not use the prototype to automatically rank, discipline,
              hire, fire or diagnose employees.
            """)

    analyze_button.click(
        fn=analyze_workforce,
        inputs=[team_dropdown],
        outputs=[
            summary_output,
            employee_table,
            chart_output,
            priority_table,
            priority_message_output
        ]
    )

    # Show a report when the interface first opens
    demo.load(
        fn=analyze_workforce,
        inputs=[team_dropdown],
        outputs=[
            summary_output,
            employee_table,
            chart_output,
            priority_table,
            priority_message_output
        ]
    )

demo.launch(share=True, debug=False)


/tmp/ipykernel_596/4262787018.py:296: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7ba57c2b577564d2a5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
